# eve-api Tutorial

A quick walkthrough of the `eve-api` package: a minimal authenticated HTTP client for the EVE API.

## Installation

```bash
cd eve-api
pip install -e .
```

## 1. Connect and log in

In [ ]:
from eve_api import EVEClient

eve = EVEClient()          # defaults to https://api.eve-chat.chat
await eve.__aenter__()     # open the HTTP connection

await eve.login("your-email@example.com", "your-password")
print("Authenticated:", eve.is_authenticated())

Authenticated: True


## 2. GET requests

All convenience methods (`get`, `post`, `patch`, `delete`) return parsed JSON as plain Python dicts/lists.

In [2]:
me = await eve.get("/users/me")
print(f"User: {me['email']}")
print(f"Name: {me.get('first_name', '')} {me.get('last_name', '')}")

User: james@trillium.tech
Name: None None


In [3]:
collections = await eve.get("/collections/public")

for col in collections["data"]:
    print(f"  - {col['name']}")

print(f"\nTotal: {collections['meta']['total_count']}")

  - Wiley AI Gateway
  - Wikipedia EO
  - EVE open access
  - ESA EO Knowledge Base

Total: 4


## 3. POST, PATCH, DELETE

In [4]:
# Create a conversation
conv = await eve.post("/conversations", json={"name": "Tutorial Chat"})
conv_id = conv["id"]
print(f"Created: {conv_id} — {conv['name']}")

Created: 698f4c7a37c7389604facd9f — Tutorial Chat


In [5]:
# Rename it
updated = await eve.patch(f"/conversations/{conv_id}", json={"name": "Renamed Chat"})
print(f"Renamed to: {updated['name']}")

Renamed to: Renamed Chat


In [6]:
# Delete it
await eve.delete(f"/conversations/{conv_id}")
print("Deleted.")

Deleted.


## 4. Streaming a response

The `stream()` method yields parsed SSE event dicts as they arrive from the server.

In [7]:
conv = await eve.post("/conversations", json={"name": "Stream Demo"})
conv_id = conv["id"]

async for event in eve.stream(
    f"/conversations/{conv_id}/stream_messages",
    json={
        "query": "What is Synthetic Aperture Radar?",
        "public_collections": ["eve-public"],
        "k": 3,
    },
):
    if event["type"] == "token":
        print(event["content"], end="", flush=True)
    elif event["type"] == "status":
        print(f"[{event['content']}]", flush=True)
    elif event["type"] == "final":
        print()  # newline after streaming completes

### **Synthetic Aperture Radar (SAR) – Definition & Overview**

**Synthetic Aperture Radar (SAR)** is a **remote sensing technology** that uses **microwave signals** to create high-resolution images of Earth's surface, regardless of weather conditions (clouds, rain, darkness). Unlike optical sensors (e.g., cameras), SAR actively emits and receives radar signals to map terrain, monitor changes, and detect objects.

---

### **Key Aspects of SAR**
1. **Active Sensing**:
   - SAR systems **emit microwave pulses** toward the Earth and **measure the reflected signals** (backscatter).
   - This allows imaging **day or night**, unlike passive optical sensors that rely on sunlight.

2. **Synthetic Aperture Principle**:
   - A **moving radar antenna** (e.g., on a satellite or aircraft) simulates a much larger antenna by combining signals from multiple positions along its flight path.
   - This **increases resolution** significantly compared to traditional radar.

3. **Wavelength Bands**:
   - S

In [8]:
# Clean up
await eve.delete(f"/conversations/{conv_id}")
print("Conversation deleted.")

Conversation deleted.


## 5. Unauthenticated requests

Use `request()` with `auth_required=False` for endpoints that do not need a token. This returns the raw `httpx.Response`.

In [9]:
resp = await eve.request("GET", "/health", auth_required=False)
print(f"Status: {resp.status_code}")
print(resp.json())

Status: 200
{'status': 'healthy'}


## 6. Direct token access

Useful if you need to pass the token to another library or service.

In [ ]:
print("Token:", eve.token[:20], "...")
print("Headers:", eve.auth_headers)

## 7. Error handling

The client raises typed exceptions for common HTTP error codes.

In [11]:
from eve_api import NotFoundError, ForbiddenError, APIError

try:
    await eve.get("/conversations/nonexistent-id")
except NotFoundError:
    print("Not found (404)")
except ForbiddenError:
    print("Forbidden (403)")
except APIError as e:
    print(f"API error {e.status_code}: {e.message}")

Not found (404)


## Clean up

In [12]:
await eve.close()
print("Connection closed.")

Connection closed.
